# PINN Inverse Heat Source (1D) — v9

$$-T'' = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

## Progress so far

| Ver | P1 T' err | P2 f L2 | Final f L2 | Issue |
|-----|-----------|---------|-----------|-------|
| v5 | 0.153 | 0.224 | 0.277 | Strong-form T'' noisy |
| v6 | 0.176 | 0.413 | 0.218 | K=10 overfits T' |
| v7 | 0.012 | 0.122 | 0.275 | P3 smoothness $\to$ parabola |
| v8 | 0.014 | **0.120** | 0.120 | Anchor too rigid, P3 frozen |

## v8 diagnosis

Phase 2 weak loss converges to 2.55e-4, but **true f gives 5.92e-4**.
The network finds a better weak-form fit than $f^*$ because $\hat{T}'$ is biased!
Phase 3 must jointly move T and f, but constant anchor prevents any movement.

## v9 strategy

1. **Decaying anchor**: $w_{anchor}(t) = w_0 \cdot \max(0, 1 - t/T_{decay})$. High initially (prevent regression), zero later (let physics take over).
2. **Reduced Tikhonov** in Phase 2: was competing with weak loss (reg~8 vs weak~3e-4 at w=1e-4, so effective reg = 8e-4 $\approx$ 3x weak!)
3. **Larger lr for f** in Phase 3: 1e-3 (was 5e-4)
4. **Higher K_TEST=15**: more test functions = better conditioning

In [ ]:
import sympy, subprocess, sys
if int(sympy.__version__.split('.')[1]) >= 14:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sympy==1.13.1'])
    print('Fixed. RESTART RUNTIME then skip this cell.')
else:
    print(f'sympy {sympy.__version__} OK')

In [ ]:
import sys, os, gc

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    RESULTS_DIR = '/content/results'
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt, matplotlib.animation as animation
torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def free_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f'PyTorch {torch.__version__} | {device}')

In [ ]:
# === Architecture (K=5) ===
class FourierFeatures(nn.Module):
    def __init__(self, K=5):
        super().__init__()
        self.K = K
        self.register_buffer('freqs', torch.arange(1, K+1, dtype=torch.float32)*np.pi)
    @property
    def out_dim(self): return 1+2*self.K
    def forward(self, x):
        return torch.cat([x, torch.sin(x*self.freqs), torch.cos(x*self.freqs)], dim=-1)

class FourierMLP(nn.Module):
    def __init__(self, hidden, K=5, scale=1.0):
        super().__init__()
        self.ff = FourierFeatures(K)
        dims = [self.ff.out_dim]+list(hidden)+[1]
        net = []
        for i in range(len(dims)-1):
            lin = nn.Linear(dims[i], dims[i+1])
            nn.init.xavier_normal_(lin.weight); lin.weight.data *= scale
            nn.init.zeros_(lin.bias); net.append(lin)
            if i < len(dims)-2: net.append(nn.Tanh())
        self.net = nn.Sequential(*net)
    def forward(self, x): return self.net(self.ff(x))

def freeze(m, flag):
    for p in m.parameters(): p.requires_grad_(flag)

K_NET = 5
torch.manual_seed(42)
pinn       = FourierMLP([64,64,64], K=K_NET, scale=1.0).to(device)
source_net = FourierMLP([64,64,64], K=K_NET, scale=0.5).to(device)
print(f'K={K_NET} | T: {sum(p.numel() for p in pinn.parameters())} | f: {sum(p.numel() for p in source_net.parameters())}')

In [ ]:
# === Data ===
def f_true(x): return np.sin(np.pi*x) + 0.5*np.sin(3*np.pi*x)
def T_true(x): return np.sin(np.pi*x)/np.pi**2 + 0.5*np.sin(3*np.pi*x)/(9*np.pi**2)

N_OBS=25; NOISE=0.01
x_obs_np = np.linspace(0.05,0.95,N_OBS)
T_clean = T_true(x_obs_np)
noise_amp = NOISE*np.max(np.abs(T_clean))
T_obs_np = T_clean + noise_amp*np.random.randn(N_OBS)
NOISE_FLOOR = noise_amp**2

x_obs = torch.tensor(x_obs_np,dtype=torch.float32).unsqueeze(1).to(device)
T_obs = torch.tensor(T_obs_np,dtype=torch.float32).unsqueeze(1).to(device)
N_COL=400
x_col = torch.linspace(0,1,N_COL).unsqueeze(1).to(device)
dx = 1.0/(N_COL-1)
x_eval = torch.linspace(0,1,500).unsqueeze(1).to(device)
x_plot = np.linspace(0,1,500)
x0=torch.zeros(1,1).to(device); x1=torch.ones(1,1).to(device)
xc = x_col.cpu().squeeze().numpy()
dTt = np.cos(np.pi*xc)/np.pi + 0.5*np.cos(3*np.pi*xc)/(3*np.pi)
print(f'{N_OBS} obs, {NOISE*100:.0f}% noise | noise floor={NOISE_FLOOR:.2e}')

In [ ]:
# === Weak-form PDE (K_TEST=15, was 8) ===
K_TEST = 15  # more test functions = better conditioned system
k_vals = torch.arange(1,K_TEST+1,dtype=torch.float32).to(device)
xcf = x_col.squeeze()
phi  = torch.sin(k_vals[:,None]*np.pi*xcf[None,:])
dphi = k_vals[:,None]*np.pi*torch.cos(k_vals[:,None]*np.pi*xcf[None,:])
trap = torch.ones(N_COL,device=device)*dx
trap[0]*=0.5; trap[-1]*=0.5

def weak_loss(dTdx, f_vals):
    dT=dTdx.squeeze(); f=f_vals.squeeze()
    lhs = (dT[None,:]*dphi*trap[None,:]).sum(1)
    rhs = (f[None,:]*phi*trap[None,:]).sum(1)
    return ((lhs-rhs)**2).mean()

def get_dTdx(model, x, cg=True):
    x_ = x.detach().clone().requires_grad_(True)
    T = model(x_)
    return torch.autograd.grad(T, x_, torch.ones_like(T), create_graph=cg)[0]

print(f'Weak-form: K_TEST={K_TEST} test functions (was 8 in v7/v8)')

## Phase 1 — T regression (proven good from v7/v8)

In [ ]:
# === Phase 1: T with smoothness + weight decay + early stop ===
freeze(pinn,True); freeze(source_net,False)
P1=100_000
opt1 = optim.Adam(pinn.parameters(), lr=1e-3, weight_decay=1e-3)
sch1 = optim.lr_scheduler.StepLR(opt1, step_size=10000, gamma=0.3)
h1=[]; best_st=None; best_v=1e9; stopped=P1

for ep in range(1,P1+1):
    opt1.zero_grad()
    Lb=pinn(x0).squeeze()**2+pinn(x1).squeeze()**2
    Ld=((pinn(x_obs)-T_obs)**2).mean()
    dT=get_dTdx(pinn,x_col)
    loss = 500*Lb+500*Ld+0.5*(dT**2).mean()
    loss.backward(); opt1.step(); sch1.step()
    h1.append({'bc':Lb.item(),'data':Ld.item(),'smooth':(dT**2).mean().item()})
    d=Ld.item()
    v=abs(np.log10(max(d,1e-20))-np.log10(NOISE_FLOOR))
    if d>NOISE_FLOOR*0.1 and v<best_v:
        best_v=v; best_st={k:v.clone() for k,v in pinn.state_dict().items()}
    if ep>5000 and d<NOISE_FLOOR*0.5:
        stopped=ep; print(f'  EARLY STOP ep {ep}'); break
    if ep%5000==0:
        dTc=get_dTdx(pinn,x_col,cg=False).detach().cpu().squeeze().numpy()
        print(f'  {ep:6d} | data {d:.3e} | dT err {np.mean((dTc-dTt)**2)**0.5:.4f}')
        free_memory()

if best_st: pinn.load_state_dict(best_st); print('  Restored best')
with torch.no_grad(): T_p1=pinn(x_eval).cpu().squeeze().numpy()
e1=np.abs(T_p1-T_true(x_plot))
dTf=get_dTdx(pinn,x_col,cg=False).detach().cpu().squeeze().numpy()
dTe=np.mean((dTf-dTt)**2)**0.5
print(f'Phase 1 done (ep {stopped}) | T L2={np.mean(e1**2)**0.5:.2e} | dT err={dTe:.4f}')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(x_plot,T_true(x_plot),'b-',lw=2.5,label='$T^*$')
axes[0].plot(x_plot,T_p1,'r--',lw=2,label='$\\hat{T}$')
axes[0].scatter(x_obs_np,T_obs_np,c='gray',s=40,zorder=5,label='Obs.')
axes[0].set_title(f'T(x) | L2={np.mean(e1**2)**0.5:.2e}'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(xc,dTt,'b-',lw=2.5,label="$T'^*$")
axes[1].plot(xc,dTf,'r--',lw=1.5,label="$\\hat{T}'$")
axes[1].set_title(f"$T'$ quality | err={dTe:.4f}"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Phase 1',fontweight='bold'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase1.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

## Phase 2 — $f$ via weak-form (reduced Tikhonov)

In v8, Tikhonov reg value was ~8 with weight 1e-4, giving effective contribution ~8e-4.
Weak loss was ~2.5e-4. So **Tikhonov was 3x stronger than physics!**

Reducing to `W2R=1e-5` (10x smaller).

In [ ]:
# === Phase 2: f via weak form (REDUCED Tikhonov) ===
freeze(pinn,False); freeze(source_net,True)
P2=100_000
W2R = 1e-5   # 10x smaller than v8! Was competing with weak loss.

dT_frozen = get_dTdx(pinn,x_col,cg=False).detach()
ft = torch.tensor(f_true(xc),dtype=torch.float32).unsqueeze(1).to(device)
target_wl = weak_loss(dT_frozen, ft).item()
print(f'Weak loss(true f) = {target_wl:.3e}')
print(f'Phase 2: {P2} ep | Tikhonov w={W2R} (was 1e-4)')

opt2 = optim.Adam(source_net.parameters(), lr=1e-3)
sch2 = optim.lr_scheduler.StepLR(opt2, step_size=15000, gamma=0.3)
h2 = []

for ep in range(1, P2+1):
    opt2.zero_grad()
    Lw = weak_loss(dT_frozen, source_net(x_col))
    xr = x_col.detach().clone().requires_grad_(True)
    df = torch.autograd.grad(source_net(xr), xr, torch.ones_like(xr), create_graph=True)[0]
    Lr = (df**2).mean()
    (Lw + W2R*Lr).backward(); opt2.step(); sch2.step()
    h2.append({'weak': Lw.item(), 'reg': Lr.item()})
    if ep%10000==0:
        with torch.no_grad():
            fp = source_net(x_eval).cpu().squeeze().numpy()
            fl2 = np.mean((fp-f_true(x_plot))**2)**0.5
        print(f'  {ep:6d} | weak {Lw.item():.3e} | reg {Lr.item():.3e} | f L2={fl2:.3e}')
        free_memory()

with torch.no_grad(): f_p2 = source_net(x_eval).cpu().squeeze().numpy()
e2 = np.abs(f_p2-f_true(x_plot))
print(f'Phase 2 done | f L2={np.mean(e2**2)**0.5:.2e}')
with torch.no_grad(): f_anchor = source_net(x_col).detach().clone()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(x_plot,f_true(x_plot),'b-',lw=2.5,label='$f^*$ true')
plt.plot(x_plot,f_p2,'r--',lw=2,label='$\\mathcal{N}_f$ Phase 2')
plt.title(f'Phase 2 | f L2={np.mean(e2**2)**0.5:.2e}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase2.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

## Phase 3 — Joint fine-tuning (DECAYING ANCHOR)

**v8 problem**: constant anchor froze f for 100k epochs.

**v9**: $w_{anchor}(ep) = w_0 \cdot \max(0,\; 1 - ep / T_{decay})$
- First 30k epochs: anchor decays linearly from 10 to 0 (safe transition)
- After 30k: no anchor, pure physics drives optimization
- f lr = 1e-3 (2x larger than v8)
- T lr = 1e-4 (2x larger, T needs to move to fix its T' bias)

In [ ]:
# === Phase 3: joint with DECAYING anchor ===
freeze(pinn,True); freeze(source_net,True)
P3 = 100_000
PDE_RAMP = 20_000
ANCHOR_DECAY = 30_000   # anchor goes to 0 over this many epochs

WP = 200.0    # PDE (high)
WB = 500.0    # BC
WD = 50.0     # Data
WR = 1e-6     # Tikhonov on f' (very light)
W_ANC0 = 10.0  # initial anchor weight

# Differential lr: T slightly larger than v8 to fix T' bias
opt3 = optim.Adam([
    {'params': pinn.parameters(), 'lr': 1e-4},        # T: can adjust to fix bias
    {'params': source_net.parameters(), 'lr': 1e-3},   # f: needs to improve
])
sch3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=P3, eta_min=1e-6)

allp = list(pinn.parameters()) + list(source_net.parameters())
h3 = []; snaps3 = []
print(f'Phase 3: {P3} ep | lr_T=1e-4, lr_f=1e-3')
print(f'  Anchor: {W_ANC0} -> 0 over {ANCHOR_DECAY} ep')
print(f'  PDE ramp: 0 -> {WP} over {PDE_RAMP} ep')

for ep in range(1, P3+1):
    opt3.zero_grad()
    
    # Schedule
    wp = WP * min(1.0, ep/PDE_RAMP)
    w_anc = W_ANC0 * max(0.0, 1.0 - ep/ANCHOR_DECAY)  # DECAYING anchor
    
    # Weak PDE
    dT = get_dTdx(pinn, x_col)
    f_col = source_net(x_col)
    Lw = weak_loss(dT, f_col)
    
    # BC + Data
    Lb = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
    Ld = ((pinn(x_obs)-T_obs)**2).mean()
    
    # Tikhonov
    xr = x_col.detach().clone().requires_grad_(True)
    df = torch.autograd.grad(source_net(xr), xr, torch.ones(N_COL,1,device=device), create_graph=True)[0]
    Lr = (df**2).mean()
    
    # Decaying anchor
    La = ((f_col - f_anchor)**2).mean()
    
    loss = wp*Lw + WB*Lb + WD*Ld + WR*Lr + w_anc*La
    loss.backward()
    torch.nn.utils.clip_grad_norm_(allp, 1.0)
    opt3.step(); sch3.step()
    
    h3.append({'weak':Lw.item(),'bc':Lb.item(),'data':Ld.item(),'reg':Lr.item(),'anchor':La.item(),'w_anc':w_anc})
    if ep%500==0 or ep==1:
        with torch.no_grad():
            snaps3.append((ep,x_eval.cpu().squeeze().numpy().copy(),
                           pinn(x_eval).cpu().squeeze().numpy().copy(),
                           source_net(x_eval).cpu().squeeze().numpy().copy()))
    if ep%10000==0:
        with torch.no_grad():
            fc=source_net(x_eval).cpu().squeeze().numpy()
            fl2=np.mean((fc-f_true(x_plot))**2)**0.5
            dTc=get_dTdx(pinn,x_col,cg=False).detach().cpu().squeeze().numpy()
            dte=np.mean((dTc-dTt)**2)**0.5
        print(f'  {ep:6d} | weak {Lw.item():.3e} | data {Ld.item():.3e} | anc {La.item():.3e} (w={w_anc:.1f}) | f L2={fl2:.3e} | dT err={dte:.4f}')
        free_memory()

with torch.no_grad():
    Ta=pinn(x_eval).cpu().squeeze().numpy()
    fa=source_net(x_eval).cpu().squeeze().numpy()
print(f'Adam | T L2={np.mean((Ta-T_true(x_plot))**2)**0.5:.2e} | f L2={np.mean((fa-f_true(x_plot))**2)**0.5:.2e}')

In [ ]:
# === Phase 3b: L-BFGS (no anchor, just physics) ===
LB = 500
olb = optim.LBFGS(allp, lr=0.1, max_iter=20, history_size=50, line_search_fn='strong_wolfe')
hlb = []
print(f'L-BFGS: {LB} steps')

for s in range(1,LB+1):
    def closure():
        olb.zero_grad()
        dT = get_dTdx(pinn,x_col)
        fc = source_net(x_col)
        Lw = weak_loss(dT,fc)
        Lb = pinn(x0).squeeze()**2+pinn(x1).squeeze()**2
        Ld = ((pinn(x_obs)-T_obs)**2).mean()
        xr = x_col.detach().clone().requires_grad_(True)
        df = torch.autograd.grad(source_net(xr),xr,torch.ones(N_COL,1,device=device),create_graph=True)[0]
        loss = WP*Lw+WB*Lb+WD*Ld+WR*(df**2).mean()
        loss.backward(); return loss
    v = olb.step(closure)
    hlb.append(v.item() if torch.is_tensor(v) else v)
    if s%100==0:
        with torch.no_grad():
            fc=source_net(x_eval).cpu().squeeze().numpy()
            fl2=np.mean((fc-f_true(x_plot))**2)**0.5
        print(f'  {s:4d} | loss {hlb[-1]:.3e} | f L2={fl2:.3e}')
        free_memory()

with torch.no_grad():
    T_final=pinn(x_eval).cpu().squeeze().numpy()
    f_final=source_net(x_eval).cpu().squeeze().numpy()
snaps3.append(('final',x_eval.cpu().squeeze().numpy(),T_final,f_final))
Te=np.abs(T_final-T_true(x_plot)); fe=np.abs(f_final-f_true(x_plot))
print(f'\nFINAL: T max={Te.max():.2e} L2={np.mean(Te**2)**0.5:.2e}')
print(f'       f max={fe.max():.2e} L2={np.mean(fe**2)**0.5:.2e}')

## Results

In [ ]:
# === Final results ===
xn=x_eval.cpu().squeeze().numpy()
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
ax.plot(x_plot,T_true(x_plot),'b-',lw=2.5,label='Analytical $T^*$')
ax.plot(xn,T_final,'r--',lw=2,label='PINN')
ax.scatter(x_obs_np,T_obs_np,c='gray',s=40,zorder=5,alpha=0.8,label='Obs.')
ax.set_title('Temperature $T(x)$',fontsize=13); ax.set_xlabel('x'); ax.legend(); ax.grid(alpha=0.3)
ax.text(0.05,0.95,f'Max:{Te.max():.2e}\nL2:{np.mean(Te**2)**0.5:.2e}',
        transform=ax.transAxes,va='top',bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.8),fontsize=10)
ax=axes[1]
ax.plot(x_plot,f_true(x_plot),'b-',lw=2.5,label='True $f^*$')
ax.plot(xn,f_final,'r--',lw=2,label='PINN final')
ax.plot(x_plot,f_p2,'g:',lw=1.5,alpha=0.6,label=f'Phase 2 (L2={np.mean(e2**2)**0.5:.3f})')
ax.set_title('Source $f(x)$ recovered',fontsize=13); ax.set_xlabel('x'); ax.legend(); ax.grid(alpha=0.3)
ax.text(0.05,0.95,f'Max:{fe.max():.2e}\nL2:{np.mean(fe**2)**0.5:.2e}',
        transform=ax.transAxes,va='top',bbox=dict(boxstyle='round',facecolor='lightblue',alpha=0.8),fontsize=10)
plt.suptitle('PINN Inverse - v9',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/final.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Diagnostics ===
fig,axes=plt.subplots(1,3,figsize=(18,4))
x_=x_col.detach().clone().requires_grad_(True)
T_=pinn(x_); dT_=torch.autograd.grad(T_,x_,torch.ones_like(T_),create_graph=True)[0]
d2T_=torch.autograd.grad(dT_,x_,torch.ones_like(dT_),create_graph=False)[0]
res=(d2T_.detach()+source_net(x_col).detach()).squeeze().cpu().numpy()
axes[0].plot(xc,res,'b-',lw=1.5); axes[0].axhline(0,color='k',ls=':',lw=0.5)
axes[0].set_title(f'Strong-form residual | rms={np.sqrt(np.mean(res**2)):.3e}'); axes[0].grid(alpha=0.3)
dTe2=get_dTdx(pinn,x_col,cg=False).detach().cpu().squeeze().numpy()
axes[1].plot(xc,dTt,'b-',lw=2.5,label="$T'^*$")
axes[1].plot(xc,dTe2,'r--',lw=1.5,label="$\\hat{T}'$ final")
axes[1].set_title(f"Final $T'$ | err={np.mean((dTe2-dTt)**2)**0.5:.4f}"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(x_plot,f_true(x_plot),'b-',lw=2.5,label='True')
axes[2].plot(x_plot,f_p2,'g--',lw=1.5,label=f'P2 (L2={np.mean(e2**2)**0.5:.3f})')
axes[2].plot(xn,f_final,'r--',lw=2,label=f'Final (L2={np.mean(fe**2)**0.5:.3f})')
axes[2].set_title('f: Phase 2 vs Final'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/diag.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
# === Loss + anchor schedule ===
fig,axes=plt.subplots(1,5,figsize=(25,4))
axes[0].semilogy([h['data'] for h in h1],label='Data',alpha=0.8)
axes[0].semilogy([h['bc'] for h in h1],label='BC',alpha=0.6)
axes[0].axhline(NOISE_FLOOR,color='red',ls=':',label='Noise')
axes[0].set_title('Phase 1'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[1].semilogy([h['weak'] for h in h2],label='Weak')
axes[1].semilogy([h['reg'] for h in h2],label='Tikh',ls='--')
axes[1].set_title('Phase 2'); axes[1].legend(); axes[1].grid(alpha=0.3)
for k,ls in [('weak','-'),('bc','--'),('data','-.'),('anchor',':')]:
    axes[2].semilogy([max(h[k],1e-20) for h in h3],ls=ls,label=k.upper(),alpha=0.8)
axes[2].set_title('Phase 3 losses'); axes[2].legend(fontsize=7); axes[2].grid(alpha=0.3)
axes[3].plot([h['w_anc'] for h in h3],'r-',lw=1.5)
axes[3].set_title('Anchor weight schedule'); axes[3].set_xlabel('Epoch'); axes[3].grid(alpha=0.3)
axes[4].semilogy(hlb,'b-',lw=1.5)
axes[4].set_title('L-BFGS'); axes[4].grid(alpha=0.3)
plt.suptitle('v9 Loss History',fontweight='bold'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Animation ===
fig,axes=plt.subplots(1,2,figsize=(11,4))
def animate(i):
    lbl,xs,Ts,fs=snaps3[i]
    for ax in axes: ax.cla()
    axes[0].plot(x_plot,T_true(x_plot),'b-',lw=2,alpha=0.5,label='$T^*$')
    axes[0].plot(xs,Ts,'r-',lw=2,label='PINN')
    axes[0].scatter(x_obs_np,T_obs_np,c='gray',s=25,zorder=5)
    axes[0].set_ylim(-0.005,0.14); axes[0].set_title(f'T ep {lbl}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x_plot,f_true(x_plot),'b-',lw=2,alpha=0.5,label='$f^*$')
    axes[1].plot(xs,fs,'r-',lw=2,label='PINN')
    axes[1].set_ylim(-0.5,1.5); axes[1].set_title(f'f ep {lbl}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
ani=animation.FuncAnimation(fig,animate,frames=len(snaps3),interval=200)
gp=f'{RESULTS_DIR}/anim.gif'
ani.save(gp,writer='pillow',fps=4,dpi=80); plt.close(); free_memory()
print(f'GIF: {os.path.getsize(gp)//1024} KB')
from IPython.display import Image; Image(gp)

## Ablations

In [ ]:
# === Noise sensitivity ===
def train_v9(xo, To, na, e1=15000, e2=30000, e3=40000, lb=200, wr2=1e-5, anc_decay=15000):
    torch.manual_seed(0)
    p=FourierMLP([64,64,64],K=K_NET,scale=1.0).to(device)
    s=FourierMLP([64,64,64],K=K_NET,scale=0.5).to(device)
    nf=max(na**2,1e-12)
    # P1
    freeze(p,True); freeze(s,False)
    o=optim.Adam(p.parameters(),lr=1e-3,weight_decay=1e-3)
    for e in range(e1):
        o.zero_grad()
        dT=get_dTdx(p,x_col)
        l=500*(p(x0).squeeze()**2+p(x1).squeeze()**2)+500*((p(xo)-To)**2).mean()+0.5*(dT**2).mean()
        l.backward(); o.step()
        if e>3000 and ((p(xo)-To)**2).mean().item()<nf*0.5: break
    # P2
    freeze(p,False); freeze(s,True)
    dTf=get_dTdx(p,x_col,cg=False).detach()
    o2=optim.Adam(s.parameters(),lr=1e-3)
    for e in range(e2):
        o2.zero_grad()
        Lw=weak_loss(dTf,s(x_col))
        xr=x_col.detach().clone().requires_grad_(True)
        df=torch.autograd.grad(s(xr),xr,torch.ones_like(xr),create_graph=True)[0]
        (Lw+wr2*(df**2).mean()).backward(); o2.step()
    with torch.no_grad(): fa=s(x_col).detach().clone()
    # P3 with decaying anchor
    freeze(p,True); freeze(s,True)
    o3=optim.Adam([{'params':p.parameters(),'lr':1e-4},{'params':s.parameters(),'lr':1e-3}])
    ap=list(p.parameters())+list(s.parameters())
    for e in range(e3):
        o3.zero_grad()
        wp=200*min(1,(e+1)/10000)
        wa=10*max(0,1-(e+1)/anc_decay)
        dT=get_dTdx(p,x_col); fc=s(x_col)
        l=wp*weak_loss(dT,fc)+500*(p(x0).squeeze()**2+p(x1).squeeze()**2)+50*((p(xo)-To)**2).mean()+wa*((fc-fa)**2).mean()
        l.backward(); torch.nn.utils.clip_grad_norm_(ap,1.0); o3.step()
    ol=optim.LBFGS(ap,lr=0.1,max_iter=20,history_size=50,line_search_fn='strong_wolfe')
    for _ in range(lb):
        def cl():
            ol.zero_grad()
            dT=get_dTdx(p,x_col); fc=s(x_col)
            l=200*weak_loss(dT,fc)+500*(p(x0).squeeze()**2+p(x1).squeeze()**2)+50*((p(xo)-To)**2).mean()
            l.backward(); return l
        ol.step(cl)
    with torch.no_grad(): fp=s(x_eval).cpu().squeeze().numpy().copy()
    del p,s; free_memory(); return fp

noises=[0.0,0.01,0.02,0.05,0.10]
rn={}; xn=x_eval.cpu().squeeze().numpy()
print('Noise sensitivity (v9)')
for n in noises:
    np.random.seed(0)
    na=n*np.max(np.abs(T_clean))
    Tn=T_clean+na*np.random.randn(N_OBS)
    Tt=torch.tensor(Tn,dtype=torch.float32).unsqueeze(1).to(device)
    fp=train_v9(x_obs,Tt,max(na,1e-10))
    l2=float(np.mean((fp-f_true(xn))**2)**0.5)
    rn[n]=(fp,l2); print(f'  {n*100:5.1f}% -> L2={l2:.4f}')

fig,axes=plt.subplots(1,2,figsize=(14,5))
cm=plt.cm.plasma
axes[0].plot(x_plot,f_true(x_plot),'k-',lw=3,label='$f^*$',zorder=10)
for i,(n,(fp,_)) in enumerate(rn.items()):
    axes[0].plot(xn,fp,'--',lw=1.5,color=cm(i/len(noises)),label=f'{n*100:.0f}%')
axes[0].set_title('Recovery vs noise'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([n*100 for n in noises],[rn[n][1] for n in noises],'ro-',ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L2'); axes[1].grid(alpha=0.3)
plt.suptitle('Noise Sensitivity (v9)',fontweight='bold'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Summary ===
import glob
print('='*55)
print('  v9 RESULTS')
print('='*55)
print(f'  T L2: {np.mean(Te**2)**0.5:.2e}')
print(f'  f L2: {np.mean(fe**2)**0.5:.2e}')
print(f'  f L2 (Phase 2): {np.mean(e2**2)**0.5:.2e}')
print(f'  K={K_NET}, K_TEST={K_TEST}')
print(f'  Phases: P1={stopped} P2={P2} P3={P3}+{LB}')
print('  Key: decaying anchor, reduced Tikhonov, diff lr')
print('='*55)
for fp in sorted(glob.glob(f'{RESULTS_DIR}/*')):
    print(f'  {os.path.basename(fp):35s} {os.path.getsize(fp)//1024:>5} KB')